In [1]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(len(text))
print(text[:300])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [3]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encodde("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]


NameError: name 'encodde' is not defined

In [4]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [5]:
print(encode("안녕"))

KeyError: '안'

In [6]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:20])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])


In [9]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

1003854 111540


In [13]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"입력이 {context.tolist()} 일 떼 -> 정답은 {target.item()}")

입력이 [18] 일 떼 -> 정답은 47
입력이 [18, 47] 일 떼 -> 정답은 56
입력이 [18, 47, 56] 일 떼 -> 정답은 57
입력이 [18, 47, 56, 57] 일 떼 -> 정답은 58
입력이 [18, 47, 56, 57, 58] 일 떼 -> 정답은 1
입력이 [18, 47, 56, 57, 58, 1] 일 떼 -> 정답은 15
입력이 [18, 47, 56, 57, 58, 1, 15] 일 떼 -> 정답은 47
입력이 [18, 47, 56, 57, 58, 1, 15, 47] 일 떼 -> 정답은 58


In [14]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1]for i in ix])
    return x, y

xb, yb = get_batch('train')
print(xb.shape, yb.shape)
print(xb)
print(yb)
print(torch.equal(xb[:, 1:], yb[:, :-1]))    

torch.Size([4, 8]) torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
True


In [17]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss) 

torch.Size([32, 65])
tensor(5.0364, grad_fn=<NllLossBackward0>)


In [18]:
def generate(model, idx, max_new_tokens):
    for _ in range(max_new_tokens):
        logits, _ = model(idx)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

start = torch.zeros((1, 1), dtype=torch.long)
print(decode(generate(model, start, 300)[0].tolist()))


lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheHd:.?nA!LgjgL&jcM-sZW!HFsWt'rv.x;z&hPyd'XHOxnAcj.erRjcLxxVFSrOpW.oeGJ Ai$yi&!mRfeopx,f.e..iBgxVrl-VThV?W?qF;roi!NxbR;zyDlfd&DH'X;PgqTOMcL
HvR;tg;tEkFST$Cil;AkkEptkGClcJ NoPTeZLPX.B-aft'ZEl'rJfEDm&UZ
RE


In [20]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
batch_size = 32

for step in range(10000):
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if step % 1000 == 0:
        print(step, loss.item())

print(loss.item())        

0 4.67774772644043
1000 3.699721097946167
2000 3.0974154472351074
3000 2.9000706672668457
4000 2.7394089698791504
5000 2.607576370239258
6000 2.3726768493652344
7000 2.4460175037384033
8000 2.460146188735962
9000 2.560112953186035
2.4238574504852295


In [21]:
print(decode(generate(model, start, 300)[0].tolist()))



Toreenday oshad tlthie cke lime te h m,

Thely isieshok thovety liss he d
Thie lorr hr, de alchak s GHAMGcatowist pst:


hat; YO, f heldver ENowhisweathascoundeangenofthare le cen My
Wir I ngd
Theintur mal, ithotche nd bupata s t whexisse ie,
LINA blfft osprise o acold te;
S:

thotou:
Miof y t amwn
